<a href="https://colab.research.google.com/github/AlejandraLopR/asistente-inteligente-llama/blob/main/Hackathon_1_Construir_un_Asistente_Inteligente_con_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Hackathon 1: Construir un Asistente Inteligente con Llama — el cierre del Módulo 1**

### 🎵 Spotify Matcher Assistant — Fine-Tuning con TinyLlama + LoRA
**Módulo 1: Hackathon de Asistentes Conversacionales**

Este cuaderno implementa un pipeline completo de fine-tuning utilizando **TinyLlama-1.1B**, **LoRA** para crear un curador musical interactivo que recomienda playlists reales de Spotify mediante un endpoint desplegado en **Gradio**.

In [ ]:
!pip install -q transformers peft accelerate datasets gradio "torchao>=0.16.0" trl

import torch
import gradio as gr
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers.utils import logging
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print(" Dependencias instaladas e importadas con éxito.")

**Módulo 2: Carga del Modelo Base y Configuración de Adaptadores LoRA**
Cargamos `TinyLlama-1.1B-Chat` en media precisión (`fp16`) e inyectamos adaptadores de bajo rango (**LoRA**).

In [ ]:
modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

modelo = AutoModelForCausalLM.from_pretrained(
    modelo_base,
    dtype=torch.float16,
    device_map="auto"
)
print("Modelo base cargado:", modelo_base)

# Configuración LoRA
config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
print(" Adaptadores LoRA listos.")

**Módulo 3: Dataset de Entrenamiento**
Construimos un dataset de recomendación musical estructurado en formato `Cliente / Agente`

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

SYSTEM_PROMPT = "<|start_header_id|>system<|end_header_id|>\n\nEres un asistente experto en recomendar playlists de Spotify.<|eot_id|>"

ejemplos_spotify = [
    {
        "texto": (
            f"{SYSTEM_PROMPT}"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            "Busco una playlist para estudiar sin distracciones.<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
            "Te recomiendo 'Deep Focus' en Spotify: ritmos Ambient para concentrarte. "
            "Escúchala aquí: https://open.spotify.com/playlist/37i9dQZF1DWZeKCadgRdKQ<|eot_id|>"
        )
    },
    {
        "texto": (
            f"{SYSTEM_PROMPT}"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            "¿Qué escucho si me gusta el Indie Rock de los 2000s?<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
            "Tu mejor opción es 'Indie Classics', con temas de The Strokes y Arctic Monkeys. "
            "Escúchala aquí: https://open.spotify.com/playlist/37i9dQZF1DX2Nc3B9P1R3A<|eot_id|>"
        )
    },
    {
        "texto": (
            f"{SYSTEM_PROMPT}"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            "Quiero música para entrenar pesado en el gimnasio.<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
            "Escucha 'Beast Mode': una mezcla potente de Hip-Hop y Rock para motivarte. "
            "Escúchala aquí: https://open.spotify.com/playlist/37i9dQZF1DX321uKiA1S2j<|eot_id|>"
        )
    },
    {
        "texto": (
            f"{SYSTEM_PROMPT}"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            "Recomiéndame algo relajante para escuchar por la tarde.<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
            "Te sugiero 'Chill Tracks': producciones suaves y R&B ideal para desconectar. "
            "Escúchala aquí: https://open.spotify.com/playlist/37i9dQZF1DX6Vq3A93I2L3<|eot_id|>"
        )
    },
    {
        "texto": (
            f"{SYSTEM_PROMPT}"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            "Busco música mexicana para una fiesta o reunión.<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
            "Te sugiero 'Fiesta Mexicana': los mejores éxitos de regional y pop para ambientar. "
            "Escúchala aquí: https://open.spotify.com/playlist/37i9dQZF1DX1lVh3P9A93I<|eot_id|>"
        )
    }
]

dataset = Dataset.from_list(ejemplos_spotify)

def tokenizar(ejemplo):
    res = tokenizer(ejemplo["texto"], truncation=True, max_length=256)
    res["labels"] = res["input_ids"].copy()
    return res

dataset_tokenizado = dataset.map(tokenizar, batched=True, remove_columns=["texto"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

**Módulo 4: Fine-Tuning Ejecutado con `Trainer`**

In [ ]:
# ==========================================
# PASO 4: ENTRENAMIENTO INTENSIVO (CONVERGENCIA RÁPIDA)
# ==========================================
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="/content/resultados_llama",
    num_train_epochs=35,             # Incrementado para asegurar la asociación directa con los links
    per_device_train_batch_size=1,
    learning_rate=3e-4,              # Tasa optimizada para adaptación rápida de LoRA
    logging_steps=1,
    report_to="none",
    fp16=True,
    optim="adamw_torch"
)

trainer = Trainer(
    model=modelo_lora,
    train_dataset=dataset_tokenizado,
    data_collator=collator,
    args=args
)

print("🚀 Iniciando re-entrenamiento...")
trainer.train()
print(" ¡Entrenamiento completado!")

**Módulo 5: Despliegue del Endpoint Interactiva en Gradio**:

Se genera una función de inferencia determinista (`do_sample=False`) y se sirve la aplicación a través de una URL pública ejecutable en tiempo real.

In [ ]:
def responder_spotify(mensaje, historial):
    prompt = (
        "<|start_header_id|>system<|end_header_id|>\n\n"
        "Eres un asistente experto en recomendar playlists de Spotify.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{mensaje}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(modelo_lora.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = modelo_lora.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    tokens_generados = outputs[0][input_length:]

    # Decodificar omitiendo tokens especiales y eliminando residuos visuales
    respuesta = tokenizer.decode(tokens_generados, skip_special_tokens=True).strip()
    respuesta = respuesta.replace("<|eot_id|>", "").strip()

    return respuesta

gr.ChatInterface(
    fn=responder_spotify,
    title="🎵 Spotify Assistant (Llama-3.2-1B + LoRA)",
    description="Asistente de recomendación de playlists en tiempo real."
).launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f7d6b5feaf1587de46.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
